# DEFHS System F — audio-native Qwen2.5-Omni-3B independent expert

This is an **experimental information-acquisition lane**, not a production System F submission.
It processes all 826 training and 209 test recordings using only the audio. It saves an append-only progress journal and never accesses test labels or submits predictions.

**Requirements:** official DataVerse competition input, Kaggle Internet enabled, one GPU. On a T4 it may exceed memory; treat out-of-memory as a technical blocker rather than a scientific negative.

The paired scientific comparator is the separate `DEFHS_SYSTEM_F_AUDIO_EXPERT_AUDIT.py`, using authenticated System E OOF probabilities. No post-hoc label overrides are authorized.

In [ ]:
#!/usr/bin/env python3
"""DEFHS independent audio-native Qwen2.5-Omni-3B expert. Never submits.

Scientific contract: one pretrained checkpoint, one unmodified 4-way prompt,
all 826 train + all 209 test samples, no use of test ground truth,
fixed deterministic greedy decoding, append-only progress.

Preflight: first 12 training recordings as technical smoke test; no accuracy gate.
No Kaggle competition predictions changed until independent nested evaluation.
"""
import os,sys,subprocess
from importlib.metadata import version, PackageNotFoundError

def installed(pkg):
 try:return version(pkg)
 except PackageNotFoundError:return None

requirements=['transformers==4.57.3','accelerate>=1.2','qwen-omni-utils','soundfile','librosa']
if installed('transformers')!='4.57.3' or installed('qwen-omni-utils') is None or installed('accelerate') is None:
 print('INSTALLING_MODEL_RUNTIME',requirements,flush=True)
 subprocess.check_call([sys.executable,'-m','pip','install','-q',*requirements])
 os.execv(sys.executable,[sys.executable,*sys.argv])
import torch, pandas as pd, numpy as np
import json, re, traceback, time, hashlib, gc
from pathlib import Path
from transformers import Qwen2_5OmniForConditionalGeneration,Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info
from sklearn.metrics import f1_score, confusion_matrix, classification_report

CLASSES=['angry','happy','neutral','sad']
MODEL='Qwen/Qwen2.5-Omni-3B'
OUT=Path('/kaggle/working/defhs_system_f_omni_audio')
OUT.mkdir(exist_ok=True,parents=True)
assert torch.cuda.is_available(), 'Enable GPU accelerator in Kaggle settings'
print('CUDA',torch.cuda.get_device_name(0),'VRAM GB',torch.cuda.get_device_properties(0).total_memory/2**30,flush=True)

# Competition input only, no external test labels or private evaluation.
candidates=[p for p in Path('/kaggle/input').rglob('train.csv') if (p.parent/'test.csv').exists()]
assert len(candidates)>=1,'Attach the official DataVerse competition input'
roots=[p.parent for p in candidates if 'dataverse' in str(p).lower() or 'srcasw' in str(p).lower()]
root=roots[0] if roots else candidates[0].parent
train=pd.read_csv(root/'train.csv');test=pd.read_csv(root/'test.csv')
assert len(train)==826 and len(test)==209
idtr='Id' if 'Id' in train.columns else 'audio_id'
idte='Id' if 'Id' in test.columns else 'audio_id'
ycol='emotion' if 'emotion' in train.columns else 'label'
train['ID']=train[idtr].astype(int);test['ID']=test[idte].astype(int)
assert len(set(train.ID)&set(test.ID))==0
assert set(train[ycol].astype(str).str.lower())==set(CLASSES)

def wav_for_id(i):
 p=list(root.rglob(f'audio_{int(i):06}.wav'))
 assert len(p)==1,(i,p)
 return str(p[0])
# technical smoke test samples taken before any model result; balanced, seed-stable.
pilot=[]
for c in CLASSES:
 pilot.extend(train.loc[train[ycol].str.lower()==c,'ID'].sort_values().tolist()[:3])
assert len(pilot)==12
remaining=[int(i) for i in train.ID if int(i) not in set(pilot)]
order=pilot+remaining+[int(i) for i in test.ID]
assert len(order)==1035 and len(set(order))==1035
labels={int(r.ID):str(getattr(r,ycol)).lower() for r in train.itertuples()}
phase={int(i):('train' if i in labels else 'test') for i in order}

SYSTEM='You listen to speech to judge vocal emotion, using pitch, energy, tempo, timbre, intonation and audible expression. The spoken words alone do not determine the vocal emotion. Answer exactly one of: angry, happy, neutral, sad. No explanation.'
QUESTION='Listen to the Hindi speech recording. What emotion is conveyed by the WAY the person speaks? Output exactly one lowercase word: angry, happy, neutral, or sad.'

print('Loading',MODEL,flush=True)
t0=time.monotonic()
processor=Qwen2_5OmniProcessor.from_pretrained(MODEL)
model=Qwen2_5OmniForConditionalGeneration.from_pretrained(
 MODEL,torch_dtype=torch.float16,device_map='auto',low_cpu_mem_usage=True,
 attn_implementation='sdpa').eval()
model.disable_talker()
print('MODEL_LOADED_SEC',round(time.monotonic()-t0,2),'GPU_ALLOC_GB',round(torch.cuda.memory_allocated()/2**30,2),flush=True)

journal=OUT/'AUDIO_LABEL_JOURNAL.jsonl'
done={}
if journal.exists():
 for line in journal.read_text().splitlines():
  if line.strip():
   v=json.loads(line);done[int(v['Id'])]=v
 assert set(done)<=set(order),'Journal contains foreign IDs'
print('RESUMED_N',len(done))

def normalize_answer(answer):
 # Greedy generation is expected to return a single label. Mark unexpected as abstention.
 s=answer.lower().strip().replace('**','').strip(' .,:;!\n\t`')
 if s in CLASSES:return s
 chunks=re.findall(r'\b(?:angry|happy|neutral|sad)\b',s)
 return chunks[-1] if len(set(chunks))==1 else 'unclear'

def run_one(i):
 path=wav_for_id(i)
 conversation=[
  {'role':'system','content':[{'type':'text','text':SYSTEM}]},
  {'role':'user','content':[{'type':'audio','audio':path},{'type':'text','text':QUESTION}]}
 ]
 templ=processor.apply_chat_template(conversation,add_generation_prompt=True,tokenize=False)
 audio,images,videos=process_mm_info(conversation,use_audio_in_video=False)
 inputs=processor(text=templ,audio=audio,images=images,videos=videos,
                  return_tensors='pt',padding=True,use_audio_in_video=False)
 inputs=inputs.to(model.device).to(model.dtype)
 with torch.inference_mode():
  result=model.generate(**inputs,return_audio=False,do_sample=False,max_new_tokens=16)
 # HuggingFace examples return full text_ids; decode only generated tail where applicable.
 token_ids=result[0]
 n=int(inputs['input_ids'].shape[-1])
 tail=token_ids[n:] if len(token_ids)>n else token_ids
 raw=processor.batch_decode([tail],skip_special_tokens=True,clean_up_tokenization_spaces=False)[0]
 return {'Id':int(i),'split':phase[i],'prediction':normalize_answer(raw),'raw':raw,
         'wav_sha256':hashlib.sha256(Path(path).read_bytes()).hexdigest()}

start=time.monotonic()
for count,i in enumerate(order,1):
 if i in done:continue
 try:
  val=run_one(i)
 except Exception as exc:
  err={'Id':int(i),'split':phase[i],'prediction':'ERROR',
       'raw':repr(exc),'traceback':traceback.format_exc(limit=4)}
  (OUT/'LAST_ERROR.json').write_text(json.dumps(err,indent=2))
  print('ERROR ON ID',i,repr(exc),flush=True)
  raise
 with journal.open('a',encoding='utf-8') as f:
  f.write(json.dumps(val,ensure_ascii=False)+'\n')
 done[i]=val
 if count<=13 or count%20==0:
  print('PROGRESS',count,'/',len(order),'ID',i,'PRED',val['prediction'],
        'SECONDS',round(time.monotonic()-start,1),flush=True)
 if count==12:
  assert sum(done[i]['prediction']!='ERROR' for i in pilot)==12
  print('PILOT_12_COMPLETED',[(int(i),done[i]['prediction']) for i in pilot],flush=True)

frame=pd.DataFrame(done.values()).sort_values('Id')
assert len(frame)==1035 and frame.Id.nunique()==1035
frame.to_csv(OUT/'OMNI_ALL1035.csv',index=False,float_format='%.12g')
matched=frame[frame.split=='train'].set_index('Id').loc[train.ID]
y=train[ycol].astype(str).str.lower().to_numpy()
pr=matched.prediction.to_numpy()
recognized=np.isin(pr,CLASSES)
report={
 'model':MODEL,'n_total':len(frame),'n_train':826,'n_test':209,
 'valid_train':int(recognized.sum()),'valid_test':int(frame.query("split=='test'").prediction.isin(CLASSES).sum()),
 'abstention_train':int((~recognized).sum()),
 'exact_greedy_audio_prompt':SYSTEM+' '+QUESTION,
 'no_kaggle_submission':True,'no_test_labels_used':True,
 'wall_seconds_this_run':round(time.monotonic()-start,2),
 'train_zero_shot_macro_f1_treat_abstain_as_miss':float(f1_score(y,pr,labels=CLASSES,average='macro',zero_division=0)),
 'confusion_valid_only':confusion_matrix(y[recognized],pr[recognized],labels=CLASSES).tolist(),
 'class_report':classification_report(y,pr,labels=CLASSES,zero_division=0,output_dict=True),
 'input_root':str(root),'all_audio_sha_verified_in_journal':True
}
(OUT/'RESULT.json').write_text(json.dumps(report,indent=2)+'\n')
print('FINAL_RESULT',json.dumps({k:v for k,v in report.items() if k not in ['confusion_valid_only','class_report','exact_greedy_audio_prompt']},indent=2),flush=True)
